# Stage 4b — Fine-tuning CodeBERT end to end

Unfreezes the encoder and trains it with the pooling head, for the two architectures Stage 4a
carried forward. The result is compared against Stage 2's reproduction of Levin —
**75.7% accuracy, κ 62.1** — on the same test commits.

| | Stage 4a validation κ | stride |
|---|---|---|
| transformer pooling | 44.1 (sd 1.3) | half overlap |
| attention pooling | 40.4 (sd 3.0) | no overlap |

**Protocol.** 5-fold cross-validation, repeated twice, on the training rows to choose between them;
then a final fit per architecture across three seeds; then the test set, opened once. Roughly
**10 hours on an A100**, so everything is checkpointed to Drive and resumes where it stopped.

**On the optimiser.** AdamW adapts the step size per parameter from gradient history — it does not
choose the learning rate. The base rate still sets the scale, and what varies it over training is
the scheduler. Both are here: discriminative rates (encoder 2e-5, head 1e-3) under linear warmup
and decay.

## 0. Setup

Run this cell first. It works in Google Colab, Jupyter and IDEs such as VS Code or PyCharm.

It finds this repository (on Colab, opened straight from GitHub, it clones it), then picks one
working folder, `WORK`, where this notebook reads its inputs and writes its outputs. Every
notebook in the repository uses the same `WORK`, so they hand files to each other through it.

- set the environment variable `CIC_WORK` to choose the folder yourself
- otherwise it is `<repository>/work`, or on Colab a folder on your Google Drive
  (set `CIC_DRIVE=0` to stay on Colab's local disk)

An input missing from `WORK` is copied from the repository or downloaded from this study's Zenodo
record or Hugging Face repository, so any notebook can be run without re-running the ones before
it. See `replication.py`.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = "https://github.com/Binamra00/commit-intent-and-refactoring-levels"
_candidates = [Path(os.environ["CIC_ROOT"])] if os.environ.get("CIC_ROOT") else []
_candidates += [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in _candidates if (p / "replication.py").exists()), None)
if ROOT is None:                      # opened on its own, e.g. from GitHub in Colab
    ROOT = Path.cwd() / REPO.rsplit("/", 1)[-1]
    if not (ROOT / "replication.py").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)
sys.path.insert(0, str(ROOT))

from replication import setup, require
WORK = setup()

## 1. Drive, and resume

Colab wipes local storage on disconnect, so on Colab `WORK` is on Drive and the checkpoints go there. Two levels:

- **per run** — a completed fold appends its score and is skipped on a re-run
- **per epoch** — a run interrupted midway resumes from its last completed epoch, rather than
  losing all three

A run is roughly 25 minutes, so the second level is what makes a dropped connection cost minutes
instead of the lot.

In [1]:
import os, json, re, math, time, gc, shutil, collections
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold
from sklearn.metrics import (accuracy_score, cohen_kappa_score, f1_score,
                             confusion_matrix)

# WORK comes from the setup cell; on Colab it is on Drive, so a disconnect loses at most
# one epoch.

RUNS  = WORK / "runs.jsonl"              # one line per completed run
EPOCH = WORK / "epoch_ckpt"              # mid-run state
EPOCH.mkdir(exist_ok=True)
print(f"checkpoints: {WORK}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}, {p.total_memory/1e9:.0f} GB")
    BF16 = torch.cuda.is_bf16_supported()
    print(f"bfloat16 autocast: {'on' if BF16 else 'unavailable, using fp32'}")
else:
    BF16 = False
    print("No GPU. This notebook needs one -- on CPU it is days, not hours.")

Mounted at /content/drive
checkpoints: /content/drive/MyDrive/levin_stage4b
GPU: NVIDIA A100-SXM4-80GB, 85 GB
bfloat16 autocast: on


In [2]:
MODEL   = "microsoft/codebert-base"
MAXLEN  = 512
LABELS  = ["adaptive", "corrective", "perfective"]
SEEDS   = [42, 1337, 2024]
EPOCHS   = 10        # 4a and the first 4b run were still improving at 3
PATIENCE = 3         # stop after this many epochs without a better validation kappa
LR_ENC, LR_HEAD = 2e-5, 1e-3
N_FOLDS, N_REPEATS = 5, 2
ARCHS = {"transformer": 256, "attention": None}      # pooling -> stride, from Stage 4a

# Inputs come from WORK; require() copies or downloads any that are missing.
SEARCH = [WORK]

def find_input(name):
    return require(name, optional=True)


need = ["levin_with_diffs.jsonl", "stage2_train_idx.npy", "stage2_test_idx.npy",
        "levin_dataset.pkl", "levin_columns.json"]
paths = {n: find_input(n) for n in need}

print("inputs\n")
for n, p in paths.items():
    print(f"   {n:<28}{p if p else 'NOT FOUND'}")

missing = [n for n, p in paths.items() if p is None]
if missing:
    print("\nsearched:")
    for d in SEARCH:
        print(f"   {d}")
    raise SystemExit(f"missing {missing} -- run Stages 1-3 first")

D = pd.read_pickle(paths["levin_dataset.pkl"])
R = pd.DataFrame([json.loads(l) for l in open(paths["levin_with_diffs.jsonl"], encoding="utf-8")
                  if l.strip()])
R = R[R["ok"] == True].reset_index(drop=True)
R["java_files"] = R["files"].map(
    lambda fs: sum(1 for f in (fs or []) if str(f.get("filename", "")).endswith(".java")))
R = R[(R["truncated"] != True) & (R["java_files"] > 0)].reset_index(drop=True)

ID_COL = json.load(open(paths["levin_columns.json"]))["hash"]
d2 = D.drop_duplicates(subset=[ID_COL], keep="first").reset_index(drop=True)
tr_ids = set(d2.iloc[np.load(paths["stage2_train_idx.npy"])][ID_COL])
te_ids = set(d2.iloc[np.load(paths["stage2_test_idx.npy"])][ID_COL])
R["split"] = R["commit_id"].map(
    lambda c: "train" if c in tr_ids else ("test" if c in te_ids else "drop"))
R = R[R.split != "drop"].reset_index(drop=True)

L2I = {l: i for i, l in enumerate(LABELS)}
Y = R["label"].map(L2I).to_numpy()
TRAIN_IDX = R.index[R.split == "train"].to_numpy()
TEST_IDX  = R.index[R.split == "test"].to_numpy()
print(f"\n{len(R):,} commits   train {len(TRAIN_IDX):,}   test {len(TEST_IDX):,}"
      f"   (test sealed until section 6)")

inputs

   levin_with_diffs.jsonl      /content/drive/MyDrive/levin_stage4b/levin_with_diffs.jsonl
   stage2_train_idx.npy        /content/drive/MyDrive/levin_stage4b/stage2_train_idx.npy
   stage2_test_idx.npy         /content/drive/MyDrive/levin_stage4b/stage2_test_idx.npy
   levin_dataset.pkl           /content/drive/MyDrive/levin_stage4b/levin_dataset.pkl
   levin_columns.json          /content/drive/MyDrive/levin_stage4b/levin_columns.json

1,150 commits   train 977   test 173   (test sealed until section 6)


## 2. Pieces, and how many fit

Same structured representation as Stage 3b and 4a.

With the encoder frozen, a 248-piece commit was 248 cheap forward passes. With gradients on, every
activation is held for the backward pass, so the largest commits set the memory ceiling. Rather
than guess a cap, the next cell probes the GPU with progressively larger commits and reports the
largest that fits.

In [3]:
HUNK = re.compile(r"^@@ .*? @@")

def message_of(rec):
    return (rec.get("git_message") or rec.get("dataset_comment") or "").strip()


def all_pieces(rec):
    """Message piece first, then one per changed file, largest first."""
    msg = f"<COMMIT_MESSAGE>{message_of(rec)}</COMMIT_MESSAGE>"
    files = []
    for f in rec.get("files") or []:
        added, removed = [], []
        for ln in (f.get("patch") or "").split("\n"):
            if HUNK.match(ln) or ln.startswith(("--- ", "+++ ")):
                continue
            if ln.startswith("+"):
                added.append(ln[1:])
            elif ln.startswith("-"):
                removed.append(ln[1:])
        piece = [f"<FILE>{f.get('filename')}</FILE>"]
        if removed:
            piece.append("<REMOVED>" + "\n".join(removed) + "</REMOVED>")
        if added:
            piece.append("<ADDED>" + "\n".join(added) + "</ADDED>")
        files.append("\n".join(piece))
    files.sort(key=len, reverse=True)
    return [msg] + files


ALL = R.apply(all_pieces, axis=1)
print(f"{int(ALL.map(len).sum()):,} pieces across {len(R):,} commits")
print(f"per commit: median {int(ALL.map(len).median())}, "
      f"90th {int(ALL.map(len).quantile(.9))}, "
      f"99th {int(ALL.map(len).quantile(.99))}, max {int(ALL.map(len).max())}")

7,181 pieces across 1,150 commits
per commit: median 3, 90th 13, 99th 43, max 248


In [4]:
tok = AutoTokenizer.from_pretrained(MODEL)

def windows(text, stride):
    ids = tok.encode(text, add_special_tokens=False, truncation=False)
    body = MAXLEN - 2
    if len(ids) <= body:
        return [ids]
    step = body if stride is None else stride
    return [ids[i:i + body] for i in range(0, len(ids), step) if i < len(ids)]


def probe_capacity(candidates=(32, 64, 128, 256), stride=256):
    """
    Largest piece cap that survives a forward+backward on this GPU.
    Uses the biggest commit in the corpus, which is the worst case by construction.
    """
    if DEVICE != "cuda":
        print("no GPU; cap left at 64"); return 64
    from transformers import AutoModel as AM
    worst = int(ALL.map(len).idxmax())
    enc = AM.from_pretrained(MODEL).to(DEVICE)
    enc.gradient_checkpointing_enable()
    head = nn.Linear(enc.config.hidden_size, len(LABELS)).to(DEVICE)
    ok = 0
    for cap in candidates:
        pieces = ALL.iloc[worst][:cap]
        wins = [w for p in pieces for w in windows(p, stride)]
        try:
            torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
            L = MAXLEN
            ids = torch.full((len(wins), L), tok.pad_token_id, dtype=torch.long)
            att = torch.zeros((len(wins), L), dtype=torch.long)
            for i, w in enumerate(wins):
                seq = [tok.cls_token_id] + w[:L-2] + [tok.sep_token_id]
                ids[i, :len(seq)] = torch.tensor(seq); att[i, :len(seq)] = 1
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=BF16):
                h = enc(input_ids=ids.to(DEVICE), attention_mask=att.to(DEVICE))
                loss = head(h.last_hidden_state[:, 0]).mean()
            loss.backward()
            peak = torch.cuda.max_memory_allocated() / 1e9
            print(f"   cap {cap:>4}: {len(wins):>5} windows, peak {peak:5.1f} GB   ok")
            ok = cap
        except torch.cuda.OutOfMemoryError:
            print(f"   cap {cap:>4}: out of memory")
            torch.cuda.empty_cache(); break
        finally:
            enc.zero_grad(set_to_none=True); head.zero_grad(set_to_none=True)
    del enc, head; gc.collect(); torch.cuda.empty_cache()
    return ok or 32


print("probing the largest workable cap on this GPU\n")
MAX_PIECES = probe_capacity()
print(f"\nMAX_PIECES = {MAX_PIECES}")

PIECES = ALL.map(lambda ps: ps[:MAX_PIECES])
capped = int((ALL.map(len) > MAX_PIECES).sum())
lost = int((ALL.map(len) - PIECES.map(len)).sum())
print(f"commits capped : {capped} of {len(R):,} ({capped/len(R)*100:.2f}%)")
print(f"pieces dropped : {lost:,} of {int(ALL.map(len).sum()):,} "
      f"({lost/ALL.map(len).sum()*100:.2f}%)")
print("""
Dropped pieces are the smallest file changes in the largest commits; the message and the biggest
changes are always kept. Report this figure with the results.""")

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

probing the largest workable cap on this GPU



... (2 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

   cap   32:    32 windows, peak   7.3 GB   ok
   cap   64:    64 windows, peak  13.8 GB   ok
   cap  128:   128 windows, peak  27.0 GB   ok
   cap  256:   248 windows, peak  51.6 GB   ok

MAX_PIECES = 256
commits capped : 0 of 1,150 (0.00%)
pieces dropped : 0 of 7,181 (0.00%)

Dropped pieces are the smallest file changes in the largest commits; the message and the biggest
changes are always kept. Report this figure with the results.


## 3. Batching by window budget

A batch of four commits could be four windows or nearly a thousand, because piece counts vary so
widely. Batching by commit count would run comfortably most of the time and fail on the tail.

Batches are therefore assembled to a **window budget** — commits are added until the total windows
would exceed it. Memory use is then roughly constant regardless of which commits land together.

In [5]:
class CommitDataset(Dataset):
    def __init__(self, idx, stride):
        self.idx = np.asarray(idx); self.stride = stride
        self.cache = {}

    def __len__(self):
        return len(self.idx)

    def n_windows(self, i):
        return len(self[i]["windows"])

    def __getitem__(self, i):
        ci = int(self.idx[i])
        if ci not in self.cache:
            wins, owner = [], []
            for pi, p in enumerate(PIECES.iloc[ci]):
                for w in windows(p, self.stride):
                    wins.append(w); owner.append(pi)
            self.cache[ci] = (wins, owner)
        wins, owner = self.cache[ci]
        return {"windows": wins, "owner": owner, "y": int(Y[ci]), "ci": ci}


class WindowBudgetSampler(Sampler):
    """Groups commits into batches whose total window count stays under a budget."""

    def __init__(self, ds, budget, shuffle=True, seed=0):
        self.ds, self.budget, self.shuffle, self.seed = ds, budget, shuffle, seed
        self.sizes = np.array([ds.n_windows(i) for i in range(len(ds))])

    def __iter__(self):
        order = np.arange(len(self.ds))
        if self.shuffle:
            rng = np.random.default_rng(self.seed); rng.shuffle(order)
        batch, used = [], 0
        for i in order:
            s = int(self.sizes[i])
            if batch and used + s > self.budget:
                yield batch; batch, used = [], 0
            batch.append(int(i)); used += s
        if batch:
            yield batch

    def __len__(self):
        return max(1, math.ceil(self.sizes.sum() / self.budget))


def collate(batch):
    flat, wowner, commit_of = [], [], []
    for b_i, b in enumerate(batch):
        for w, o in zip(b["windows"], b["owner"]):
            flat.append(w); wowner.append(o); commit_of.append(b_i)
    L = min(MAXLEN, max(len(w) for w in flat) + 2)
    ids = torch.full((len(flat), L), tok.pad_token_id, dtype=torch.long)
    att = torch.zeros((len(flat), L), dtype=torch.long)
    for i, w in enumerate(flat):
        seq = [tok.cls_token_id] + w[:L - 2] + [tok.sep_token_id]
        ids[i, :len(seq)] = torch.tensor(seq); att[i, :len(seq)] = 1
    return {"ids": ids, "att": att, "commit_of": torch.tensor(commit_of),
            "piece_of": torch.tensor(wowner),
            "y": torch.tensor([b["y"] for b in batch]), "n_commits": len(batch)}


WINDOW_BUDGET = 96 if DEVICE == "cuda" else 8
print(f"window budget per batch: {WINDOW_BUDGET}")
print("Raise it if the GPU has headroom -- larger batches are faster and steadier;")
print("lower it if section 5 runs out of memory.")

window budget per batch: 96
Raise it if the GPU has headroom -- larger batches are faster and steadier;
lower it if section 5 runs out of memory.


## 4. Model

The encoder is shared across a commit's pieces. Windows are mean-pooled within their piece first,
so a long file gets one vote rather than several, then the pieces are pooled into a commit vector
by whichever strategy is being tested.

In [6]:
class CommitClassifier(nn.Module):
    def __init__(self, pooling, dropout=0.2):
        super().__init__()
        self.enc = AutoModel.from_pretrained(MODEL)
        self.enc.gradient_checkpointing_enable()
        hid = self.enc.config.hidden_size
        self.pooling = pooling
        if pooling == "attention":
            self.score = nn.Linear(hid, 1)
        else:
            layer = nn.TransformerEncoderLayer(hid, nhead=4, dim_feedforward=512,
                                               dropout=dropout, batch_first=True)
            self.tr = nn.TransformerEncoder(layer, num_layers=1)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(hid, len(LABELS))

    def forward(self, ids, att, commit_of, piece_of, n_commits):
        h = self.enc(input_ids=ids, attention_mask=att).last_hidden_state[:, 0]

        key = commit_of * 100_000 + piece_of
        uniq, inv = torch.unique(key, return_inverse=True)
        pv = torch.zeros(len(uniq), h.size(1), device=h.device, dtype=h.dtype)
        pv.index_add_(0, inv, h)
        cnt = torch.zeros(len(uniq), device=h.device, dtype=h.dtype)
        cnt.index_add_(0, inv, torch.ones(len(h), device=h.device, dtype=h.dtype))
        pv = pv / cnt.unsqueeze(1)
        pc = (uniq // 100_000).long()

        per = torch.bincount(pc, minlength=n_commits)
        P = int(per.max().item())
        x = torch.zeros(n_commits, P, h.size(1), device=h.device, dtype=h.dtype)
        m = torch.zeros(n_commits, P, device=h.device, dtype=h.dtype)
        slot = torch.zeros(n_commits, dtype=torch.long, device=h.device)
        for i in range(len(pc)):
            c = pc[i]; x[c, slot[c]] = pv[i]; m[c, slot[c]] = 1; slot[c] += 1

        mm = m.unsqueeze(-1)
        if self.pooling == "attention":
            a = self.score(x).masked_fill(mm == 0, -1e9).softmax(1)
            v = (x * a).sum(1)
        else:
            hh = self.tr(x, src_key_padding_mask=(m == 0))
            v = (hh * mm).sum(1) / mm.sum(1).clamp(min=1)
        return self.out(self.drop(v))


def class_weights(rows):
    c = np.bincount(Y[rows], minlength=len(LABELS))
    return torch.tensor(c.sum() / (len(LABELS) * c), dtype=torch.float32)


print("model defined: shared encoder, gradient checkpointing, two poolings")

model defined: shared encoder, gradient checkpointing, two poolings


### Training, with resume at epoch granularity

In [7]:
completed = {}
if RUNS.exists():
    for line in open(RUNS, encoding="utf-8"):
        if line.strip():
            r = json.loads(line); completed[r["key"]] = r
    print(f"{len(completed)} completed run(s) in WORK; they will be skipped")


def evaluate(model, loader):
    model.eval(); P, T = [], []
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16,
                                         enabled=(BF16 and DEVICE == "cuda")):
        for b in loader:
            lg = model(b["ids"].to(DEVICE), b["att"].to(DEVICE),
                       b["commit_of"].to(DEVICE), b["piece_of"].to(DEVICE), b["n_commits"])
            P.append(lg.float().argmax(1).cpu().numpy()); T.append(b["y"].numpy())
    return np.concatenate(P), np.concatenate(T)


def train_one(pooling, stride, fit_rows, val_rows, seed, key, keep_model=False):
    """Three epochs, resuming from the last completed one if interrupted."""
    torch.manual_seed(seed); np.random.seed(seed)
    model = CommitClassifier(pooling).to(DEVICE)

    enc_p  = [p for n, p in model.named_parameters() if n.startswith("enc.")]
    head_p = [p for n, p in model.named_parameters() if not n.startswith("enc.")]
    opt = torch.optim.AdamW([{"params": enc_p, "lr": LR_ENC},
                             {"params": head_p, "lr": LR_HEAD}], weight_decay=0.01)

    ds_fit = CommitDataset(fit_rows, stride)
    ds_val = CommitDataset(val_rows, stride)
    dl_fit = DataLoader(ds_fit, batch_sampler=WindowBudgetSampler(ds_fit, WINDOW_BUDGET,
                                                                  seed=seed),
                        collate_fn=collate)
    dl_val = DataLoader(ds_val, batch_sampler=WindowBudgetSampler(ds_val, WINDOW_BUDGET,
                                                                  shuffle=False),
                        collate_fn=collate)
    steps = len(dl_fit) * EPOCHS
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    lossf = nn.CrossEntropyLoss(weight=class_weights(fit_rows).to(DEVICE))

    ck = EPOCH / f"{key.replace('|', '_')}.pt"
    start, best_k, best_state, bad = 0, -1.0, None, 0
    if ck.exists():
        st = torch.load(ck, map_location=DEVICE)
        model.load_state_dict(st["model"]); opt.load_state_dict(st["opt"])
        sched.load_state_dict(st["sched"])
        start, best_k, best_state = st["epoch"] + 1, st["best_k"], st["best_state"]
        bad = st.get("bad", 0)
        print(f"      resuming {key} at epoch {start + 1} "
              f"(best {best_k:.1f}, {bad} epoch(s) without improvement)")

    for ep in range(start, EPOCHS):
        model.train(); t0 = time.time()
        for i, b in enumerate(dl_fit):
            with torch.autocast("cuda", dtype=torch.bfloat16,
                                enabled=(BF16 and DEVICE == "cuda")):
                lg = model(b["ids"].to(DEVICE), b["att"].to(DEVICE),
                           b["commit_of"].to(DEVICE), b["piece_of"].to(DEVICE), b["n_commits"])
                loss = lossf(lg.float(), b["y"].to(DEVICE))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
            if i % 25 == 0:
                print(f"\r      {key} ep{ep+1} {i}/{len(dl_fit)} loss {loss.item():.3f}",
                      end="", flush=True)

        p, t = evaluate(model, dl_val)
        k = cohen_kappa_score(t, p) * 100
        print(f"\r      {key} ep{ep+1}  val kappa {k:5.1f}  acc {accuracy_score(t,p)*100:5.1f}"
              f"  {(time.time()-t0)/60:.1f} min" + " " * 15)
        if k > best_k:
            best_k, bad = k, 0
            best_state = {n: v.detach().cpu().clone() for n, v in model.state_dict().items()}
        else:
            bad += 1
        torch.save({"epoch": ep, "model": model.state_dict(), "opt": opt.state_dict(),
                    "sched": sched.state_dict(), "best_k": best_k,
                    "best_state": best_state, "bad": bad}, ck)
        if bad >= PATIENCE:
            print(f"      early stop at epoch {ep+1}: {PATIENCE} epochs without a better "
                  f"kappa (best {best_k:.1f})")
            break

    ck.unlink(missing_ok=True)
    if keep_model:
        model.load_state_dict(best_state)
        return model, best_k
    del model; gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return None, best_k

## 5. Selection — 5-fold cross-validation, repeated twice

Twenty runs: five folds, two repeats, two architectures. At the observed 0.6 to 0.9 minutes per
epoch, ten epochs each is roughly two to three hours, less where early stopping fires. Safe to
interrupt at any point.

In [8]:
rows = []
for pooling, stride in ARCHS.items():
    rskf = RepeatedStratifiedKFold(n_splits=N_FOLDS, n_repeats=N_REPEATS,
                                   random_state=SEEDS[0])
    ks = []
    for i, (a, b) in enumerate(rskf.split(TRAIN_IDX, Y[TRAIN_IDX])):
        rep, fold = i // N_FOLDS, i % N_FOLDS
        key = f"cv|{pooling}|r{rep}f{fold}"
        if key in completed:
            ks.append(completed[key]["kappa"]); continue
        print(f"\n   {pooling}  repeat {rep+1}/{N_REPEATS}  fold {fold+1}/{N_FOLDS}")
        _, k = train_one(pooling, stride, TRAIN_IDX[a], TRAIN_IDX[b], SEEDS[0], key)
        ks.append(k)
        with open(RUNS, "a", encoding="utf-8") as f:
            f.write(json.dumps({"key": key, "kappa": k}) + "\n")
    rows.append({"pooling": pooling, "stride": "half overlap" if stride else "no overlap",
                 "cv kappa": round(float(np.mean(ks)), 1),
                 "cv sd": round(float(np.std(ks)), 1), "runs": len(ks)})

CV = pd.DataFrame(rows).sort_values("cv kappa", ascending=False)
print(f"\n\n{N_FOLDS}-FOLD CV x {N_REPEATS} REPEATS, TRAINING ROWS ONLY\n")
print(CV.to_string(index=False))

gap = CV.iloc[0]["cv kappa"] - CV.iloc[-1]["cv kappa"]
spread = CV["cv sd"].max()
print(f"\ngap {gap:.1f}   largest fold spread {spread:.1f}")
FINAL = list(ARCHS) if gap <= spread else [CV.iloc[0]["pooling"]]
print(f"taking to the test set: {FINAL}")
if gap <= spread:
    print("   The gap is within fold-to-fold variation, so both are evaluated and both reported.")


   transformer  repeat 1/2  fold 1/5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (555 > 512). Running this sequence through the model will result in indexing errors
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


      cv|transformer|r0f0 ep1  val kappa   0.0  acc  43.4  0.9 min               
      cv|transformer|r0f0 ep2  val kappa  25.2  acc  54.6  0.9 min               
      cv|transformer|r0f0 ep3  val kappa  70.2  acc  80.6  0.9 min               
      cv|transformer|r0f0 ep4  val kappa  75.1  acc  84.2  0.9 min               
      cv|transformer|r0f0 ep5  val kappa  72.4  acc  82.1  0.9 min               
      cv|transformer|r0f0 ep6  val kappa  73.5  acc  82.7  0.9 min               
      cv|transformer|r0f0 ep7  val kappa  76.6  acc  84.7  0.9 min               
      cv|transformer|r0f0 ep8  val kappa  75.5  acc  84.2  0.9 min               
      cv|transformer|r0f0 ep9  val kappa  76.3  acc  84.7  0.9 min               
      cv|transformer|r0f0 ep10  val kappa  76.3  acc  84.7  0.9 min               
      early stop at epoch 10: 3 epochs without a better kappa (best 76.6)

   transformer  repeat 1/2  fold 2/5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

      cv|transformer|r0f1 ep1  val kappa   0.0  acc  21.4  0.9 min               
      cv|transformer|r0f1 ep2  val kappa  14.2  acc  37.8  0.9 min               
      cv|transformer|r0f1 ep3  val kappa  16.2  acc  42.3  0.9 min               
      cv|transformer|r0f1 ep4  val kappa  28.4  acc  52.0  0.9 min               
      cv|transformer|r0f1 ep5  val kappa  38.5  acc  61.7  0.9 min               
      cv|transformer|r0f1 ep6  val kappa  61.2  acc  75.5  0.9 min               
      cv|transformer|r0f1 ep7  val kappa  66.1  acc  78.6  0.9 min               
      cv|transformer|r0f1 ep8  val kappa  64.5  acc  77.6  0.9 min               
      cv|transformer|r0f1 ep9  val kappa  64.0  acc  77.0  0.9 min               
      cv|transformer|r0f1 ep10  val kappa  62.5  acc  76.0  0.9 min               
      early stop at epoch 10: 3 epochs without a better kappa (best 66.1)

   transformer  repeat 1/2  fold 3/5


... (32 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

      cv|attention|r1f3 ep1  val kappa  24.3  acc  52.3  0.6 min               
      cv|attention|r1f3 ep2  val kappa  64.9  acc  77.4  0.6 min               
      cv|attention|r1f3 ep3  val kappa  64.2  acc  77.4  0.6 min               
      cv|attention|r1f3 ep4  val kappa  63.6  acc  76.4  0.6 min               
      cv|attention|r1f3 ep5  val kappa  68.2  acc  79.5  0.6 min               
      cv|attention|r1f3 ep6  val kappa  67.8  acc  79.5  0.6 min               
      cv|attention|r1f3 ep7  val kappa  67.4  acc  79.0  0.6 min               
      cv|attention|r1f3 ep8  val kappa  67.9  acc  79.5  0.6 min               
      early stop at epoch 8: 3 epochs without a better kappa (best 68.2)

   attention  repeat 2/2  fold 5/5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

      cv|attention|r1f4 ep1  val kappa   7.1  acc  37.9  0.6 min               
      cv|attention|r1f4 ep2  val kappa  56.2  acc  71.3  0.6 min               
      cv|attention|r1f4 ep3  val kappa  69.2  acc  80.0  0.6 min               
      cv|attention|r1f4 ep4  val kappa  71.1  acc  81.0  0.6 min               
      cv|attention|r1f4 ep5  val kappa  66.0  acc  77.9  0.6 min               
      cv|attention|r1f4 ep6  val kappa  68.2  acc  79.5  0.6 min               
      cv|attention|r1f4 ep7  val kappa  67.5  acc  79.0  0.6 min               
      early stop at epoch 7: 3 epochs without a better kappa (best 71.1)


5-FOLD CV x 2 REPEATS, TRAINING ROWS ONLY

    pooling       stride  cv kappa  cv sd  runs
  attention   no overlap      70.0    3.7    10
transformer half overlap      68.7    4.7    10

gap 1.3   largest fold spread 4.7
taking to the test set: ['transformer', 'attention']
   The gap is within fold-to-fold variation, so both are evaluated and both reported.


## 6. Final fits, then the test set opened once

In [9]:
for pooling in FINAL:
    for seed in SEEDS:
        key = f"final|{pooling}|{seed}"
        path = WORK / f"model_{pooling}_{seed}.pt"
        if key in completed and path.exists():
            continue
        print(f"\n   final fit: {pooling}, seed {seed}")
        a, b = train_test_split(TRAIN_IDX, test_size=0.1, random_state=seed,
                                stratify=Y[TRAIN_IDX])
        model, k = train_one(pooling, ARCHS[pooling], a, b, seed, key, keep_model=True)
        torch.save(model.state_dict(), path)
        with open(RUNS, "a", encoding="utf-8") as f:
            f.write(json.dumps({"key": key, "kappa": k}) + "\n")
        del model; gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
print("\nall final models trained")


   final fit: transformer, seed 42


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

      final|transformer|42 ep1  val kappa   0.0  acc  34.7  1.0 min               
      final|transformer|42 ep2  val kappa   0.0  acc  34.7  1.0 min               
      final|transformer|42 ep3  val kappa   1.0  acc  42.9  1.0 min               
      final|transformer|42 ep4  val kappa  51.0  acc  67.3  1.0 min               
      final|transformer|42 ep5  val kappa  60.5  acc  74.5  1.0 min               
      final|transformer|42 ep6  val kappa  66.5  acc  78.6  1.0 min               
      final|transformer|42 ep7  val kappa  61.3  acc  75.5  1.0 min               
      final|transformer|42 ep8  val kappa  66.3  acc  78.6  1.0 min               
      final|transformer|42 ep9  val kappa  64.9  acc  77.6  1.0 min               
      early stop at epoch 9: 3 epochs without a better kappa (best 66.5)

   final fit: transformer, seed 1337


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

      final|transformer|1337 ep1  val kappa   5.6  acc  45.9  1.0 min               
      final|transformer|1337 ep2  val kappa  38.8  acc  61.2  1.0 min               
      final|transformer|1337 ep3  val kappa  61.4  acc  75.5  1.0 min               
      final|transformer|1337 ep4  val kappa  64.0  acc  77.6  1.0 min               
      final|transformer|1337 ep5  val kappa  64.3  acc  77.6  1.0 min               
      final|transformer|1337 ep6  val kappa  67.6  acc  79.6  1.0 min               
      final|transformer|1337 ep7  val kappa  67.9  acc  79.6  1.0 min               
      final|transformer|1337 ep8  val kappa  69.5  acc  80.6  1.0 min               
      final|transformer|1337 ep9  val kappa  74.4  acc  83.7  1.0 min               
      final|transformer|1337 ep10  val kappa  74.4  acc  83.7  1.0 min               

   final fit: transformer, seed 2024


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

      final|attention|1337 ep1  val kappa  10.6  acc  46.9  0.7 min               
      final|attention|1337 ep2  val kappa  40.7  acc  61.2  0.7 min               
      final|attention|1337 ep3  val kappa  47.1  acc  67.3  0.7 min               
      final|attention|1337 ep4  val kappa  57.6  acc  73.5  0.7 min               
      final|attention|1337 ep5  val kappa  60.7  acc  75.5  0.7 min               
      final|attention|1337 ep6  val kappa  67.8  acc  79.6  0.7 min               
      final|attention|1337 ep7  val kappa  66.4  acc  78.6  0.7 min               
      final|attention|1337 ep8  val kappa  68.0  acc  79.6  0.7 min               
      final|attention|1337 ep9  val kappa  66.6  acc  78.6  0.7 min               
      final|attention|1337 ep10  val kappa  66.6  acc  78.6  0.7 min               

   final fit: attention, seed 2024


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

      final|attention|2024 ep1  val kappa   3.8  acc  35.7  0.7 min               
      final|attention|2024 ep2  val kappa  46.1  acc  64.3  0.7 min               
      final|attention|2024 ep3  val kappa  57.3  acc  72.4  0.7 min               
      final|attention|2024 ep4  val kappa  60.6  acc  74.5  0.7 min               
      final|attention|2024 ep5  val kappa  61.7  acc  75.5  0.7 min               
      final|attention|2024 ep6  val kappa  57.9  acc  73.5  0.7 min               
      final|attention|2024 ep7  val kappa  52.5  acc  69.4  0.7 min               
      final|attention|2024 ep8  val kappa  50.9  acc  68.4  0.7 min               
      early stop at epoch 8: 3 epochs without a better kappa (best 61.7)

all final models trained


In [10]:
results = []
for pooling in FINAL:
    for seed in SEEDS:
        path = WORK / f"model_{pooling}_{seed}.pt"
        if not path.exists():
            continue
        model = CommitClassifier(pooling).to(DEVICE)
        model.load_state_dict(torch.load(path, map_location=DEVICE))
        ds = CommitDataset(TEST_IDX, ARCHS[pooling])
        dl = DataLoader(ds, batch_sampler=WindowBudgetSampler(ds, WINDOW_BUDGET, shuffle=False),
                        collate_fn=collate)
        p, t = evaluate(model, dl)
        results.append({"pooling": pooling, "seed": seed,
                        "accuracy": round(accuracy_score(t, p) * 100, 1),
                        "kappa": round(cohen_kappa_score(t, p) * 100, 1),
                        "macro f1": round(f1_score(t, p, average="macro") * 100, 1),
                        "pred": p, "true": t})
        del model; gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

TT = pd.DataFrame([{k: v for k, v in r.items() if k not in ("pred", "true")} for r in results])
print("TEST SET, per seed\n")
print(TT.to_string(index=False))
print("\n\nMEAN ACROSS SEEDS\n")
print(TT.groupby("pooling")[["accuracy", "kappa", "macro f1"]]
      .agg(["mean", "std"]).round(1).to_string())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

TEST SET, per seed

    pooling  seed  accuracy  kappa  macro f1
transformer    42      83.8   74.6      81.7
transformer  1337      86.7   79.2      85.6
transformer  2024      82.7   73.0      82.0
  attention    42      85.5   77.1      83.4
  attention  1337      86.7   79.2      85.3
  attention  2024      87.9   81.0      86.5


MEAN ACROSS SEEDS

            accuracy      kappa      macro f1     
                mean  std  mean  std     mean  std
pooling                                           
attention       86.7  1.2  79.1  2.0     85.1  1.6
transformer     84.4  2.1  75.6  3.2     83.1  2.2


In [11]:
best = TT.groupby("pooling")["kappa"].mean().idxmax()
pick = [r for r in results if r["pooling"] == best]
p = pick[int(np.argmax([r["kappa"] for r in pick]))]

cm = confusion_matrix(p["true"], p["pred"], labels=range(len(LABELS)))
print(f"CONFUSION MATRIX -- {best}, seed {p['seed']}\n")
print(pd.DataFrame(cm.T, index=[f"classified {l}" for l in LABELS],
                   columns=[f"true {l}" for l in LABELS]).to_string())

rec = cm.diagonal() / cm.sum(axis=1) * 100
pre = cm.diagonal() / cm.sum(axis=0) * 100
print("\n\nPER CLASS, ours vs Stage 2\n")
print(pd.DataFrame({"recall ours": rec.round(0), "recall levin": [70, 87, 66],
                    "precision ours": pre.round(0), "precision levin": [65, 79, 78]},
                   index=LABELS).to_string())

ours_k = float(TT.groupby("pooling")["kappa"].mean().max())
ours_a = float(TT.groupby("pooling")["accuracy"].mean().max())
LEV_K, LEV_A, FROZEN = 62.1, 75.7, 44.1

print("\n" + "=" * 70)
print("STAGE 4B RESULT")
print("=" * 70)
print(f"   frozen encoder (4a)   kappa {FROZEN:5.1f}")
print(f"   fine-tuned  (4b)      kappa {ours_k:5.1f}   ({ours_k-FROZEN:+.1f} from unfreezing)")
print(f"   Levin    (Stage 2)    kappa {LEV_K:5.1f}   ({ours_k-LEV_K:+.1f})")
print(f"   accuracy: ours {ours_a:.1f}%   Levin {LEV_A:.1f}%")
print()
if ours_k >= LEV_K:
    print("CodeBERT on message and diff matches or beats the ChangeDistiller feature set on the")
    print("same commits -- so the diff can stand in for the AST change taxonomy, which is what")
    print("makes the method usable on a corpus where ChangeDistiller cannot run.")
elif ours_k >= LEV_K - 5:
    print("Close but behind. On 1,151 examples that is the expected shape. Worth trying before")
    print("conceding: more epochs, a larger window budget, or the full message rather than the")
    print("first line.")
else:
    print("Clearly behind. On this much data the engineered features win, and that is a finding")
    print("worth reporting rather than tuning away: for classifying these commits, Levin's")
    print("method is the better tool.")

json.dump({"cv": CV.to_dict("records"),
           "test": [{k: v for k, v in r.items() if k not in ("pred", "true")} for r in results],
           "mean_kappa": ours_k, "mean_accuracy": ours_a,
           "levin_stage2": {"accuracy": LEV_A, "kappa": LEV_K},
           "frozen_4a_kappa": FROZEN,
           "config": {"epochs": EPOCHS, "lr_encoder": LR_ENC, "lr_head": LR_HEAD,
                      "seeds": SEEDS, "max_pieces": MAX_PIECES,
                      "folds": N_FOLDS, "repeats": N_REPEATS,
                      "window_budget": WINDOW_BUDGET,
                      "commits_capped": capped, "pieces_dropped": lost}},
          open(WORK / "stage4b_results.json", "w"), indent=1)
print(f"\n   {WORK/'stage4b_results.json'} written")

CONFUSION MATRIX -- attention, seed 2024

                       true adaptive  true corrective  true perfective
classified adaptive               27                1                2
classified corrective              0               67                1
classified perfective             10                7               58


PER CLASS, ours vs Stage 2

            recall ours  recall levin  precision ours  precision levin
adaptive           73.0            70            90.0               65
corrective         89.0            87            99.0               79
perfective         95.0            66            77.0               78

STAGE 4B RESULT
   frozen encoder (4a)   kappa  44.1
   fine-tuned  (4b)      kappa  79.1   (+35.0 from unfreezing)
   Levin    (Stage 2)    kappa  62.1   (+17.0)
   accuracy: ours 86.7%   Levin 75.7%

CodeBERT on message and diff matches or beats the ChangeDistiller feature set on the
same commits -- so the diff can stand in for the AST change taxonomy, wh